# Freddie audit follow-up: payment-date gaps and two incomplete histories

Run this standalone notebook from the project root or `notebooks`, using the project's
`.venv` kernel. The earlier audit notebook does not need to be running.

The uploaded review bundle confirmed 450,000 preserved first-payment dates, 437,668
eligible loans, and four 2019 raw histories. Two checks remain:

1. Count first-payment dates after origination-quarter end for all sampled loans and
   eligible loans, with descriptive groups beyond six and twelve months.
2. Identify the two eligible 2015-H2 loans with missing monthly observations and
   inspect their raw performance records, missing months, and observed event labels.

Quarter end is a bound, not the exact origination date. Small gaps can be ordinary
payment timing; larger gaps do not identify a cause. These are descriptive groups,
not exclusion rules. The raw files are current snapshots, not historical releases.

This notebook preserves existing data, target definitions, models and notebooks.
It writes a new timestamped folder and review ZIP. It does not approve a retraining policy.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, zipfile
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
DATA = ROOT / 'data' / 'processed' / 'Freddie_Mac'
RAW = ROOT / 'data' / 'raw'
YEARS = list(range(2015, 2024))
KEYS = ['SOURCE YEAR', 'LOAN IDENTIFIER']
TARGET = 'TARGET_COMPOSITE_CREDIT_EVENT'
DECISION = pd.Period('2023-01', freq='M')
CANDIDATE_FILES = {}  # Select explicitly if multiple copies exist: {2019: 'data/processed/...parquet'}
RAW_ORIGINATION_FILES = {}  # Override raw source paths if needed.
RAW_PERFORMANCE_FILES = {}  # Override the 2019 performance source if needed.
SCAN_2019_RAW_PERFORMANCE = True  # One streamed scan, only four loan IDs retained.
FOCUS_IDS = ['F19Q20416210','F19Q20416236','F19Q40547031','F19Q40547111']
inputs = {}

def require(ok, message):
    if not ok: raise ValueError(message)

def fingerprint(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def pick(explicit, matches, label):
    if explicit is not None:
        p=Path(explicit);p=p if p.is_absolute() else ROOT/p
        require(p.is_file(), f'{label}: configured file missing: {p}')
    else:
        matches=sorted(set(Path(p).resolve() for p in matches))
        require(len(matches)==1, f'{label}: choose one exact file in configuration; found {matches}')
        p=matches[0]
    inputs[str(p.resolve())]=fingerprint(p)
    return p

require((ROOT/'src'/'freddie_performance.py').is_file(), 'Run from the project root or notebooks folder.')
model_path=DATA/'freddie_modeling_dataset_2015_2023_24m.parquet'
inputs[str(model_path.resolve())]=fingerprint(model_path)
model=pd.read_parquet(model_path, columns=KEYS+[TARGET])
require(len(model)==437668 and not model.duplicated(KEYS).any(), 'Existing modeling population differs.')
print('Audit only. No existing data, models, targets or notebook files will be modified.')


Audit only. No existing data, models, targets or notebook files will be modified.


## Reconcile the existing population and count payment-date gaps

Read existing origination and target-audit files. If more than one candidate file is
found for a year, select its exact path in `CANDIDATE_FILES` above and rerun.


In [2]:
parts=[]
for year in YEARS:
    op=DATA/f'origination_{year}_clean.parquet'
    inputs[str(op.resolve())]=fingerprint(op)
    orig=pd.read_parquet(op, columns=['LOAN IDENTIFIER','FIRST PAYMENT DATE'])
    orig['SOURCE YEAR']=year
    orig['LOAN IDENTIFIER']=orig['LOAN IDENTIFIER'].astype('string')
    ap=pick(CANDIDATE_FILES.get(year),DATA.rglob(f'performance_target_candidates_{year}.parquet'),f'{year} candidate audit')
    audit=pd.read_parquet(ap)
    audit=audit.loc[audit['HORIZON_MONTHS'].eq(24)].copy()
    audit['SOURCE YEAR']=year
    audit['LOAN IDENTIFIER']=audit['LOAN IDENTIFIER'].astype('string')
    for col in ['TARGET_ELIGIBLE','TERMINATION_WITHIN_HORIZON']:
        require(audit[col].notna().all() and audit[col].isin([True,False,0,1]).all(),f'Invalid {col}')
        audit[col]=audit[col].astype(bool)
    merged=orig.merge(audit,on=KEYS,how='outer',validate='one_to_one',indicator=True)
    require(len(merged)==50000 and merged['_merge'].eq('both').all(),f'{year}: origin/audit keys differ')
    merged=merged.drop(columns='_merge')
    text=merged['FIRST PAYMENT DATE'].astype('string').str.strip()
    require(text.str.fullmatch(r'\d{6}',na=False).all(),f'{year}: invalid payment-date format')
    merged['payment_month']=pd.to_datetime(text,format='%Y%m',errors='raise').dt.to_period('M')
    q=pd.to_numeric(merged['LOAN IDENTIFIER'].str.extract(r'^[FA]\d{2}Q([1-4])',expand=False),errors='raise')
    require(q.isin([1,2,3,4]).all(),f'{year}: invalid ID quarters')
    merged['period']=str(year)+'-'+np.where(q.le(2),'H1','H2')
    merged['cohort_end']=pd.PeriodIndex([f'{year}-'+('06' if half=='H1' else '12') for half in np.where(q.le(2),'H1','H2')],freq='M')
    merged['payment_months_after_cohort_end']=[a.ordinal-b.ordinal for a,b in zip(merged['payment_month'],merged['cohort_end'])]
    merged['full_window_end']=merged['payment_month']+23
    merged['full_window_ready_proxy']=merged['payment_month']+24
    parts.append(merged)
audit_all=pd.concat(parts,ignore_index=True)
eligible=audit_all.loc[audit_all['TARGET_ELIGIBLE']].copy()
existing=pd.MultiIndex.from_frame(model[KEYS]);observed=pd.MultiIndex.from_frame(eligible[KEYS])
require(len(existing)==len(observed) and existing.isin(observed).all() and observed.isin(existing).all(), 'Eligible population changed')
target_check=eligible[KEYS+[TARGET]].merge(model,on=KEYS,validate='one_to_one',suffixes=('_audit','_model'))
require(target_check[f'{TARGET}_audit'].eq(target_check[f'{TARGET}_model']).all(),'Candidate labels differ from modeling labels')
last=pd.to_numeric(eligible['LAST_MONTH_WITHIN_HORIZON'],errors='raise')
require(last.notna().all() and last.between(0,23).all() and last.eq(np.floor(last)).all(),'Invalid last in-window offsets')
end=np.where(eligible['TERMINATION_WITHIN_HORIZON'],last,23).astype(int)
eligible['completion_ready_proxy']=pd.PeriodIndex([p+int(o)+1 for p,o in zip(eligible['payment_month'],end)],freq='M')
eligible['expected_observations_to_endpoint']=end+1
eligible['unobserved_months_to_endpoint']=eligible['expected_observations_to_endpoint']-pd.to_numeric(eligible['OBSERVED_MONTHS'],errors='raise')
require(eligible['unobserved_months_to_endpoint'].ge(0).all(),'Observed month counts exceed endpoint span')
eligible['has_unobserved_months']=eligible['unobserved_months_to_endpoint'].gt(0)
eligible['negative_with_unobserved_months']=eligible[TARGET].eq(0)&eligible['has_unobserved_months']
eligible['ready_by_decision']=eligible['completion_ready_proxy'].le(DECISION)
cohort_summary=eligible.groupby('period').agg(
    loans=('LOAN IDENTIFIER','size'),events=(TARGET,'sum'),
    latest_first_payment=('payment_month','max'),latest_label_proxy=('completion_ready_proxy','max'),
    labels_ready=('ready_by_decision','sum'),
    loans_with_unobserved_months=('has_unobserved_months','sum'),
    negative_labels_with_unobserved_months=('negative_with_unobserved_months','sum'))
cohort_summary['labels_not_ready']=cohort_summary['loans']-cohort_summary['labels_ready']
cohort_summary['negative_gap_pct_of_cohort']=100*cohort_summary['negative_labels_with_unobserved_months']/cohort_summary['loans']
display(cohort_summary)
focus=eligible.loc[eligible['LOAN IDENTIFIER'].isin(FOCUS_IDS)].copy()
require(set(focus['LOAN IDENTIFIER'])==set(FOCUS_IDS),'Four focus loans not found in eligible population')
focus_cols=KEYS+['period','FIRST PAYMENT DATE','payment_month','full_window_end','full_window_ready_proxy',
                'completion_ready_proxy','TERMINATION_WITHIN_HORIZON','FIRST_OBSERVED_MONTH',
                'LAST_MONTH_WITHIN_HORIZON','OBSERVED_MONTHS',TARGET,'ready_by_decision']
display(focus[focus_cols])
print('This does not identify the cause of a late first payment or approve any exclusion.')

# Count first-payment gaps separately from first performance reporting.
# Quarter end gives a conservative bound: exact origination month is unavailable.
gap_parts=[]
for view,frame in [('all_sampled_originations',audit_all),('eligible_modeling_loans',eligible)]:
    g=frame.copy()
    quarter=pd.to_numeric(g['LOAN IDENTIFIER'].str.extract(r'^[FA]\d{2}Q([1-4])',expand=False),errors='raise')
    g['origination_quarter_end']=pd.PeriodIndex([
        f'{int(y)}-{int(q)*3:02d}' for y,q in zip(g['SOURCE YEAR'],quarter)],freq='M')
    g['payment_months_after_quarter_end']=[a.ordinal-b.ordinal for a,b in zip(g['payment_month'],g['origination_quarter_end'])]
    g['payment_after_quarter_end']=g['payment_months_after_quarter_end'].gt(0)
    g['payment_over_6_months_after_quarter_end']=g['payment_months_after_quarter_end'].gt(6)
    g['payment_over_12_months_after_quarter_end']=g['payment_months_after_quarter_end'].gt(12)
    g['population_view']=view
    gap_parts.append(g)
payment_gap_records=pd.concat(gap_parts,ignore_index=True)
payment_gap_summary=payment_gap_records.groupby(['population_view','period']).agg(
    loans=('LOAN IDENTIFIER','size'),
    first_payment_after_origination_quarter_end=('payment_after_quarter_end','sum'),
    first_payment_over_6_months_after_quarter_end=('payment_over_6_months_after_quarter_end','sum'),
    first_payment_over_12_months_after_quarter_end=('payment_over_12_months_after_quarter_end','sum'),
    maximum_payment_months_after_quarter_end=('payment_months_after_quarter_end','max'))
for col in ['first_payment_after_origination_quarter_end','first_payment_over_6_months_after_quarter_end','first_payment_over_12_months_after_quarter_end']:
    payment_gap_summary[col+'_pct']=100*payment_gap_summary[col]/payment_gap_summary['loans']
display(payment_gap_summary)
late_payment_records=payment_gap_records.loc[
    payment_gap_records['population_view'].eq('all_sampled_originations') &
    payment_gap_records['payment_over_6_months_after_quarter_end'],
    KEYS+['period','payment_month','origination_quarter_end','payment_months_after_quarter_end','TARGET_ELIGIBLE',TARGET]].copy()
print('Six- and twelve-month groups describe gaps; they are not proposed exclusion rules.')
print('A date after quarter end can reflect ordinary payment timing; no cause is inferred.')


,loans,events,latest_first_payment,latest_label_proxy,labels_ready,loans_with_unobserved_months,negative_labels_with_unobserved_months,labels_not_ready,negative_gap_pct_of_cohort
period,,,,,,,,,
2015-H1,23602,87,2016-12,2018-09,23602,0,0,0,0.0
2015-H2,23642,161,2017-05,2019-05,23642,2,1,0,0.00423
2016-H1,24039,157,2020-10,2022-10,24039,0,0,0,0.0
2016-H2,24037,167,2018-07,2020-07,24037,0,0,0,0.0
2017-H1,24091,222,2020-05,2022-05,24091,0,0,0,0.0
2017-H2,24187,169,2020-03,2022-03,24187,0,0,0,0.0
2018-H1,23699,327,2020-01,2021-11,23699,0,0,0,0.0
2018-H2,23718,895,2020-12,2022-12,23718,0,0,0,0.0
2019-H1,24326,1106,2021-03,2023-03,24324,0,0,2,0.0


,SOURCE YEAR,LOAN IDENTIFIER,period,FIRST PAYMENT DATE,payment_month,full_window_end,full_window_ready_proxy,completion_ready_proxy,TERMINATION_WITHIN_HORIZON,FIRST_OBSERVED_MONTH,LAST_MONTH_WITHIN_HORIZON,OBSERVED_MONTHS,TARGET_COMPOSITE_CREDIT_EVENT,ready_by_decision
224997,2019,F19Q20416210,2019-H1,202102,2021-02,2023-01,2023-02,2023-02,False,-1,23.0,24,0,False
224999,2019,F19Q20416236,2019-H1,202103,2021-03,2023-02,2023-03,2023-03,False,-1,23.0,24,0,False
249998,2019,F19Q40547031,2019-H2,202102,2021-02,2023-01,2023-02,2023-02,False,0,23.0,24,0,False
249999,2019,F19Q40547111,2019-H2,202106,2021-06,2023-05,2023-06,2023-06,False,0,23.0,24,0,False


This does not identify the cause of a late first payment or approve any exclusion.


loans  \
population_view          period           
all_sampled_originations 2015-H1  25000   
                         2015-H2  25000   
                         2016-H1  25000   
                         2016-H2  25000   
                         2017-H1  25000   
                         2017-H2  25000   
                         2018-H1  25000   
                         2018-H2  25000   
                         2019-H1  25000   
                         2019-H2  25000   
                         2020-H1  25000   
                         2020-H2  25000   
                         2021-H1  25000   
                         2021-H2  25000   
                         2022-H1  25000   
                         2022-H2  25000   
                         2023-H1  25000   
                         2023-H2  25000   
eligible_modeling_loans  2015-H1  23602   
                         2015-H2  23642   
                         2016-H1  24039   
                         2016-H2  24037   
                         2017-H1  24091   
                         2017-H2  24187   
                         2018-H1  23699   
                         2018-H2  23718   
                         2019-H1  24326   
                         2019-H2  24285   
                         2020-H1  24841   
                         2020-H2  24824   
                         2021-H1  24714   
                         2021-H2  24776   
                         2022-H1  24633   
                         2022-H2  24753   
                         2023-H1  24757   
                         2023-H2  24744   

                                  first_payment_after_origination_quarter_end  \
population_view          period                                                 
all_sampled_originations 2015-H1                                        17642   
                         2015-H2                                        16298   
                         2016-H1                                        17643   
                         2016-H2                                        16904   
                         2017-H1                                        17192   
                         2017-H2                                        16255   
                         2018-H1                                        17396   
                         2018-H2                                        15980   
                         2019-H1                                        17667   
                         2019-H2                                        16572   
                         2020-H1                                        18286   
                         2020-H2                                        16796   
                         2021-H1                                        17213   
                         2021-H2                                        16662   
                         2022-H1                                        16332   
                         2022-H2                                        15983   
                         2023-H1                                        18124   
                         2023-H2                                        16001   
eligible_modeling_loans  2015-H1                                        16619   
                         2015-H2                                        15425   
                         2016-H1                                        16991   
                         2016-H2                                        16289   
                         2017-H1                                        16560   
                         2017-H2                                        15746   
                         2018-H1                                        16501   
                         2018-H2                                        15228   
                         2019-H1                                        17240   
                         2019-H2                  

Six- and twelve-month groups describe gaps; they are not proposed exclusion rules.
A date after quarter end can reflect ordinary payment timing; no cause is inferred.


## Inspect the two histories with unobserved target-window months

The prior bundle reports two eligible 2015-H2 histories, including one negative label.
An observed positive event remains evidence of an event despite a gap. An absence of
observed events does not establish what happened during an unobserved month.

The scan below retains only the affected loans. It reports every missing month and
observations next to each gap. It reconciles observed events and counts against the
saved targets; it does not infer or replace a label in an unobserved month.


In [3]:
gap_cases=eligible.loc[eligible['has_unobserved_months']].copy()
require(len(gap_cases)==2 and gap_cases['period'].eq('2015-H2').all()
        and gap_cases[TARGET].eq(0).sum()==1,
        'Gap cases differ from the uploaded review: stop and review this output.')
gap_columns=KEYS+['period','payment_month',TARGET,'FIRST_OBSERVED_MONTH',
                 'LAST_MONTH_WITHIN_HORIZON','OBSERVED_MONTHS',
                 'TERMINATION_WITHIN_HORIZON','unobserved_months_to_endpoint']
display(gap_cases[gap_columns])

raw_parts=[]
for year,group in gap_cases.groupby('SOURCE YEAR'):
    year=int(year)
    rp=pick(RAW_PERFORMANCE_FILES.get(year),
            [*RAW.rglob(f'sample_perf_{year}.txt'),*RAW.rglob(f'sample_svcg_{year}.txt')],
            f'{year} raw performance')
    with rp.open(encoding='utf-8-sig') as stream:
        require(len(stream.readline().rstrip('\r\n').split('|'))==35,
                f'{year}: expected the existing 35-field raw performance layout.')
    ids=set(group['LOAN IDENTIFIER'])
    for chunk in pd.read_csv(rp,sep='|',header=None,usecols=[0,1,3,4,7,8,9],
                             dtype='string',keep_default_na=False,
                             chunksize=250000,encoding='utf-8-sig'):
        chunk.columns=['LOAN IDENTIFIER','PERIOD','DELINQUENCY','LOAN AGE',
                       'MODIFICATION FLAG','ZERO BALANCE CODE','ZERO BALANCE EFFECTIVE DATE']
        selected=chunk.loc[chunk['LOAN IDENTIFIER'].isin(ids)].copy()
        if not selected.empty:
            selected['SOURCE YEAR']=year
            raw_parts.append(selected)
require(bool(raw_parts),'Affected loans were not found in raw monthly records.')
raw_gap=pd.concat(raw_parts,ignore_index=True)
require(not raw_gap.duplicated(KEYS+['PERIOD']).any(),'Duplicate raw monthly records.')
require(set(raw_gap['LOAN IDENTIFIER'])==set(gap_cases['LOAN IDENTIFIER']),
        'Not all affected loans have raw histories.')
raw_gap['report_month']=pd.to_datetime(raw_gap['PERIOD'],format='%Y%m',errors='raise').dt.to_period('M')
payments=gap_cases.set_index('LOAN IDENTIFIER')['payment_month']
raw_gap['month_offset']=[p.ordinal-payments.loc[k].ordinal
                        for k,p in zip(raw_gap['LOAN IDENTIFIER'],raw_gap['report_month'])]
status=raw_gap['DELINQUENCY'].str.strip()
raw_gap['serious_event']=pd.to_numeric(status.where(status.ne('RA')),errors='coerce').ge(3)|status.eq('RA')
raw_gap['terminal_credit_event']=raw_gap['ZERO BALANCE CODE'].str.strip().str.zfill(2).isin(['02','03','09','15'])

raw_summaries=[]
missing_rows=[]
context_parts=[]
for _,loan in gap_cases.iterrows():
    part=raw_gap.loc[raw_gap['LOAN IDENTIFIER'].eq(loan['LOAN IDENTIFIER'])].sort_values('report_month')
    window=part.loc[part['month_offset'].between(0,23)]
    endpoint=int(loan['LAST_MONTH_WITHIN_HORIZON']) if loan['TERMINATION_WITHIN_HORIZON'] else 23
    observed=set(window['month_offset'].astype(int))
    missing=sorted(set(range(endpoint+1))-observed)
    event=bool((window['serious_event']|window['terminal_credit_event']).any())
    raw_summaries.append({
        'SOURCE YEAR':int(loan['SOURCE YEAR']),'LOAN IDENTIFIER':loan['LOAN IDENTIFIER'],
        'period':loan['period'],'recorded_first_payment':str(loan['payment_month']),
        'raw_first_report':str(part['report_month'].min()),
        'raw_observed_months_in_window':len(observed),'saved_observed_months':int(loan['OBSERVED_MONTHS']),
        'count_matches_saved':bool(len(observed)==int(loan['OBSERVED_MONTHS'])),
        'raw_missing_months_to_endpoint':len(missing),
        'saved_unobserved_months':int(loan['unobserved_months_to_endpoint']),
        'missing_count_matches_saved':bool(len(missing)==int(loan['unobserved_months_to_endpoint'])),
        'raw_observed_event':event,'saved_target':int(loan[TARGET]),
        'observed_event_matches_saved':bool(event==int(loan[TARGET])),
        'interpretation':('Observed event supports positive target; missing months remain unobserved.'
                          if event else 'No observed event; negative target retains uncertainty in missing months.')})
    for offset in missing:
        missing_rows.append({'SOURCE YEAR':int(loan['SOURCE YEAR']),
            'LOAN IDENTIFIER':loan['LOAN IDENTIFIER'],'month_offset':offset,
            'missing_calendar_month':str(loan['payment_month']+offset)})
    neighbors={offset+shift for offset in missing for shift in [-1,1]}
    context_parts.append(part.loc[part['month_offset'].isin(neighbors)])
raw_gap_summary=pd.DataFrame(raw_summaries)
missing_months=pd.DataFrame(missing_rows)
gap_context=pd.concat(context_parts,ignore_index=True)
display(raw_gap_summary)
display(missing_months)
display(gap_context)
raw_reconciliation_ok=bool(raw_gap_summary[
    ['count_matches_saved','missing_count_matches_saved','observed_event_matches_saved']].all().all())
print('Raw count/event reconciliation:',raw_reconciliation_ok)
print('A matched negative target is not proof of no event during an unobserved month.')


,SOURCE YEAR,LOAN IDENTIFIER,period,payment_month,TARGET_COMPOSITE_CREDIT_EVENT,FIRST_OBSERVED_MONTH,LAST_MONTH_WITHIN_HORIZON,OBSERVED_MONTHS,TERMINATION_WITHIN_HORIZON,unobserved_months_to_endpoint
31826,2015,F15Q30201969,2015-H2,2015-10,0,-1,23.0,23,False,1
39161,2015,F15Q40044673,2015-H2,2015-12,1,-1,23.0,22,True,2


,SOURCE YEAR,LOAN IDENTIFIER,period,recorded_first_payment,raw_first_report,raw_observed_months_in_window,saved_observed_months,count_matches_saved,raw_missing_months_to_endpoint,saved_unobserved_months,missing_count_matches_saved,raw_observed_event,saved_target,observed_event_matches_saved,interpretation
0,2015,F15Q30201969,2015-H2,2015-10,2015-09,23,23,True,1,1,True,False,0,True,No observed event; negative target retains unc...
1,2015,F15Q40044673,2015-H2,2015-12,2015-11,22,22,True,2,2,True,True,1,True,Observed event supports positive target; missi...


,SOURCE YEAR,LOAN IDENTIFIER,month_offset,missing_calendar_month
0,2015,F15Q30201969,14,2016-12
1,2015,F15Q40044673,18,2017-06
2,2015,F15Q40044673,19,2017-07


,LOAN IDENTIFIER,PERIOD,DELINQUENCY,LOAN AGE,MODIFICATION FLAG,ZERO BALANCE CODE,ZERO BALANCE EFFECTIVE DATE,SOURCE YEAR,report_month,month_offset,serious_event,terminal_credit_event
0,F15Q30201969,201611,00,14,,,,2015,2016-11,13,False,False
1,F15Q30201969,201701,00,16,,,,2015,2017-01,15,False,False
2,F15Q40044673,201705,12,18,,,,2015,2017-05,17,True,False
3,F15Q40044673,201708,15,21,,,,2015,2017-08,20,True,False


Raw count/event reconciliation: True
A matched negative target is not proof of no event during an unobserved month.


## Export for review

The bundle includes the date-gap counts, both affected loan records, raw evidence
within their target windows, missing months and adjacent records. Review this evidence
before deciding on documentation or any sensitivity analysis. No labels are changed.


In [4]:
require(all(fingerprint(Path(p))==h for p,h in inputs.items()),'An input changed during this audit.')
out=DATA/'project_timing_target_followup'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
tables={
    'first_payment_gap_counts_by_cohort.csv':payment_gap_summary.reset_index(),
    'first_payment_over_6_months_after_quarter_end.csv':late_payment_records,
    'cohort_timing_and_gap_summary.csv':cohort_summary.reset_index(),
    'two_gap_loan_records.csv':gap_cases[gap_columns],
    'two_gap_raw_reconciliation.csv':raw_gap_summary,
    'missing_calendar_months.csv':missing_months,
    'raw_observations_next_to_gaps.csv':gap_context,
    'two_gap_raw_target_window_records.csv':raw_gap.loc[raw_gap['month_offset'].between(0,23)]}
for name,frame in tables.items():frame.to_csv(out/name,index=False)
spec={'purpose':'Complete all-cohort payment-gap counts and two incomplete-history traces',
      'existing_labels_changed':False,'models_fitted':False,
      'raw_reconciliation_ok':raw_reconciliation_ok,
      'gap_loans':len(gap_cases),'negative_gap_loans':int(gap_cases[TARGET].eq(0).sum()),
      'gap_reference':'Origination-quarter end; exact origination month is not available.',
      'gap_thresholds':'Descriptive groups only; no exclusion or retraining rule approved.',
      'source_fingerprints':inputs,
      'limits':['Missing monthly outcomes remain unknown.',
                'Current raw snapshots do not verify historical publication dates.',
                'Late payment dates do not identify modification/conversion status.']}
(out/'followup_specification.json').write_text(json.dumps(spec,indent=2),encoding='utf-8')
bundle=out/'freddie_timing_target_followup_review.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(out.iterdir()):
        if p.suffix in ['.csv','.json']:archive.write(p,arcname=p.name)
with zipfile.ZipFile(bundle) as archive:
    require(archive.testzip() is None and len(archive.infolist())==9,'Review ZIP verification failed.')
print('Share this review bundle:',bundle)
print('STOP. Review the evidence before changing labels, eligibility or models.')


Share this review bundle: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\project_timing_target_followup\20261022T224334363390Z\freddie_timing_target_followup_review.zip
STOP. Review the evidence before changing labels, eligibility or models.
